# Tabela 8 — Logit Multinomial
## Forma de pagamento do exame preventivo (Papanicolau)
### PNS 2013 e 2019 (Pooled)

---

**Objetivo:** Estimar um modelo Logit Multinomial para analisar os determinantes da forma de pagamento do exame preventivo do colo do útero (Papanicolau) em mulheres brasileiras.

**Abordagem:** Utilização de abstração semântica (variáveis do `mapping.py`), sem referência direta a códigos físicos da PNS.

**Estrutura do Notebook:**
1. Importações
2. Carregamento dos dados
3. Construção da variável dependente (multinomial)
4. Construção das variáveis explicativas
5. Diagnóstico pré-modelo
6. Estimação do Logit Multinomial (ÚNICO MODELO)
7. Resultados: Tabela 8A (Coeficientes e RRR)
8. Resultados: Tabela 8B (Efeitos Marginais Médios)
9. Estatísticas globais do modelo
10. Exportação de resultados

## 1. IMPORTAÇÕES

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import sys
from matplotlib import pyplot as plt

sys.path.append("..")
from service.pns_service import get_dataframe

# Configurações para melhor legibilidade
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
np.set_printoptions(suppress=True)

## 2. CARREGAMENTO DOS DADOS (ABSTRAÇÃO SEMÂNTICA)

Usamos **exclusivamente variáveis semânticas** definidas em `mapping.py`.
Nenhum código físico da PNS aparece neste notebook.

In [2]:
# Variáveis semânticas necessárias
variaveis_necessarias = [
    "idade",
    "vive_com_companheiro",
    "tem_filhos_nascidos_vivos",
    "trabalha",
    "escolaridade_nivel",
    "renda_domiciliar_pc",
    "situacao_censitaria",
    "preventivo_quando",
    "preventivo_sus",
    "preventivo_plano",
    "preventivo_pago",
]

# Carregamento dos dados
print("\n" + "="*70)
print("CARREGAMENTO DOS DADOS")
print("="*70)

df_bruto = get_dataframe(
    variables=variaveis_necessarias,
    sources=["2013", "2019"],
    filters={
        "sexo": "2",                              # Apenas mulheres
        "idade": {"operador": ">=", "valor": 25}  # Idade >= 25 anos
    }
)

print(f"\n✓ Amostra bruta: {len(df_bruto):,} observações")
print(f"✓ Período: 2013 e 2019 (pooled)")
print(f"\nDimensões: {df_bruto.shape}")
print(f"\nPrimeiras 5 linhas:")
print(df_bruto.head())


CARREGAMENTO DOS DADOS

✓ Amostra bruta: 158,912 observações
✓ Período: 2013 e 2019 (pooled)

Dimensões: (158912, 15)

Primeiras 5 linhas:
   idade   id_upa  renda_domiciliar_pc id_domicilio preventivo_sus  \
0     49  1100001                878.0         0001              1   
1     30  1100001                300.0         0002           None   
2     29  1100001                174.0         0003              1   
3     41  1100001                650.0         0009              1   
4     38  1100001                700.0         0010           None   

  vive_com_companheiro preventivo_quando  tem_filhos_nascidos_vivos  \
0                    1                 2                        3.0   
1                    1                 5                        1.0   
2                    1                 3                        3.0   
3                    1                 2                        2.0   
4                    1              None                        NaN   

  preventivo

## 3. CONSTRUÇÃO DA VARIÁVEL DEPENDENTE (MULTINOMIAL)

Codifica mutuamente exclusivamente a forma de realização do exame preventivo:
- **y = 0:** Realizado pelo SUS (referência)
- **y = 1:** Coberto por plano de saúde privado
- **y = 2:** Pago diretamente pela mulher

In [3]:
def build_multinomial_outcome_variable(df_entrada):
    """Constrói variável dependente multinomial com validação rigorosa."""
    
    df_limpo = df_entrada.copy()
    n_inicio = len(df_limpo)
    
    print("\n" + "="*70)
    print("CONSTRUÇÃO DA VARIÁVEL DEPENDENTE")
    print("="*70)
    
    # Passo 1: Excluir mulheres que nunca fizeram o exame
    print("\n[Passo 1] Filtrar por realização do exame...")
    df_limpo = df_limpo[df_limpo["preventivo_quando"] != "5"].copy()
    n_apos_filtro = len(df_limpo)
    print(f"  → Removidas (nunca fizeram): {n_inicio - n_apos_filtro}")
    print(f"  → Amostra restante: {n_apos_filtro:,}")
    
    # Passo 2: Standardizar valores binárias
    print("\n[Passo 2] Standardizar valores binárias...")
    
    for var in ["preventivo_sus", "preventivo_plano", "preventivo_pago"]:
        df_limpo[var] = (
            df_limpo[var]
            .astype(str).str.strip().str.lower()
            .map({"1": 1, "2": 0, "sim": 1, "não": 0, "nao": 0})
        )
    
    print("  ✓ Variáveis binárias standardizadas (0/1)")
    
    # Passo 3: Codificar categorias mutuamente exclusivas
    print("\n[Passo 3] Codificar forma de pagamento (categorias exclusivas)...")
    
    df_limpo["y_forma_pagamento"] = np.nan
    
    # Categoria 0: SUS
    idx_sus = (df_limpo["preventivo_sus"] == 1) & \
              (df_limpo["preventivo_plano"] == 0) & \
              (df_limpo["preventivo_pago"] == 0)
    df_limpo.loc[idx_sus, "y_forma_pagamento"] = 0
    
    # Categoria 1: Plano
    idx_plano = (df_limpo["preventivo_plano"] == 1) & \
                (df_limpo["preventivo_sus"] == 0) & \
                (df_limpo["preventivo_pago"] == 0)
    df_limpo.loc[idx_plano, "y_forma_pagamento"] = 1
    
    # Categoria 2: Pagou
    idx_pago = (df_limpo["preventivo_pago"] == 1) & \
               (df_limpo["preventivo_sus"] == 0) & \
               (df_limpo["preventivo_plano"] == 0)
    df_limpo.loc[idx_pago, "y_forma_pagamento"] = 2
    
    print(f"  → SUS: {idx_sus.sum():,}")
    print(f"  → Plano: {idx_plano.sum():,}")
    print(f"  → Pagou: {idx_pago.sum():,}")
    
    # Passo 4: Remover observações inconsistentes
    print("\n[Passo 4] Remover observações inconsistentes...")
    n_apos_cod = len(df_limpo)
    df_limpo = df_limpo.dropna(subset=["y_forma_pagamento"])
    n_final = len(df_limpo)
    print(f"  → Removidas (inconsistências): {n_apos_cod - n_final}")
    print(f"  → Amostra final: {n_final:,}")
    
    df_limpo["y_forma_pagamento"] = df_limpo["y_forma_pagamento"].astype(int)
    
    print("\n✓ Variável dependente construída com sucesso!")
    print("\nDistribuição:")
    for cat, label in enumerate(["SUS", "Plano", "Pagou"]):
        freq = (df_limpo["y_forma_pagamento"] == cat).sum()
        pct = 100 * freq / n_final
        print(f"  {cat} ({label}): {freq:,} ({pct:.1f}%)")
    
    return df_limpo, df_limpo["y_forma_pagamento"]


df_limpo, y_forma_pagamento = build_multinomial_outcome_variable(df_bruto)


CONSTRUÇÃO DA VARIÁVEL DEPENDENTE

[Passo 1] Filtrar por realização do exame...
  → Removidas (nunca fizeram): 6799
  → Amostra restante: 152,113

[Passo 2] Standardizar valores binárias...
  ✓ Variáveis binárias standardizadas (0/1)

[Passo 3] Codificar forma de pagamento (categorias exclusivas)...
  → SUS: 14,431
  → Plano: 5,961
  → Pagou: 3,320

[Passo 4] Remover observações inconsistentes...
  → Removidas (inconsistências): 128401
  → Amostra final: 23,712

✓ Variável dependente construída com sucesso!

Distribuição:
  0 (SUS): 14,431 (60.9%)
  1 (Plano): 5,961 (25.1%)
  2 (Pagou): 3,320 (14.0%)


## 4. CONSTRUÇÃO DAS VARIÁVEIS EXPLICATIVAS

Construção robusta de variáveis explicativas com transparência total.

In [4]:
def build_explanatory_variables(df_entrada):
    """Constrói matriz de variáveis explicativas robustamente."""
    
    df_vars = df_entrada.copy()
    
    print("\n" + "="*70)
    print("CONSTRUÇÃO DE VARIÁVEIS EXPLICATIVAS")
    print("="*70)
    
    # 1. FAIXAS ETÁRIAS
    print("\n[1] Faixas etárias...")
    df_vars["idade"] = pd.to_numeric(df_vars["idade"], errors="coerce")
    
    df_vars["idade_25_34"] = ((df_vars["idade"] >= 25) & (df_vars["idade"] <= 34)).astype(int)
    df_vars["idade_35_44"] = ((df_vars["idade"] >= 35) & (df_vars["idade"] <= 44)).astype(int)
    df_vars["idade_45_54"] = ((df_vars["idade"] >= 45) & (df_vars["idade"] <= 54)).astype(int)
    df_vars["idade_65_mais"] = (df_vars["idade"] >= 65).astype(int)
    # Ref: 55-64 anos (omitida)
    
    print(f"  25-34: {df_vars['idade_25_34'].sum()}; 35-44: {df_vars['idade_35_44'].sum()}")
    print(f"  45-54: {df_vars['idade_45_54'].sum()}; 65+: {df_vars['idade_65_mais'].sum()}")
    
    # 2. ESTRUTURA FAMILIAR
    print("\n[2] Estrutura familiar...")
    
    df_vars["vive_com_companheiro"] = (
        df_vars["vive_com_companheiro"].astype(str).str.strip().str.lower()
        .map({"1": 1, "2": 0, "sim": 1, "não": 0, "nao": 0})
    )
    
    df_vars["tem_filhos_nascidos_vivos"] = pd.to_numeric(
        df_vars["tem_filhos_nascidos_vivos"], errors="coerce"
    )
    df_vars["tem_filhos"] = (df_vars["tem_filhos_nascidos_vivos"] > 0).astype(int)
    
    print(f"  Vive com companheiro: {df_vars['vive_com_companheiro'].sum()}")
    print(f"  Tem filhos: {df_vars['tem_filhos'].sum()}")
    
    # 3. TRABALHO
    print("\n[3] Inserção no mercado de trabalho...")
    
    df_vars["trabalha"] = (
        df_vars["trabalha"].astype(str).str.strip().str.lower()
        .map({"1": 1, "2": 0, "sim": 1, "não": 0, "nao": 0})
    )
    
    print(f"  Trabalha (ocupada): {df_vars['trabalha'].sum()}")
    
    # 4. ESCOLARIDADE
    print("\n[4] Escolaridade...")
    
    mapa_esc = {1: 0, 2: 4, 3: 8, 4: 11, 5: 15, 6: 18}
    df_vars["anos_estudo"] = pd.to_numeric(
        df_vars["escolaridade_nivel"], errors="coerce"
    ).map(mapa_esc)
    
    print(f"  Média: {df_vars['anos_estudo'].mean():.2f} anos")
    
    # 5. LOCALIZAÇÃO
    print("\n[5] Localização...")
    
    df_vars["urbano"] = (
        df_vars["situacao_censitaria"].astype(str).str.strip() == "1"
    ).astype(int)
    
    print(f"  Urbana: {df_vars['urbano'].sum()}")
    
    # 6. RENDA (DECIS)
    print("\n[6] Decis de renda...")
    
    df_vars["renda_domiciliar_pc"] = pd.to_numeric(
        df_vars["renda_domiciliar_pc"], errors="coerce"
    )
    
    # Detectar coluna de ano
    ano_col = None
    for col in ["origem", "ano", "year"]:
        if col in df_vars.columns:
            ano_col = col
            break
    
    if ano_col:
        df_vars["decil_renda"] = (
            df_vars.groupby(ano_col)["renda_domiciliar_pc"]
            .transform(lambda x: pd.qcut(x, 10, labels=False, duplicates="drop") + 1)
        )
    else:
        df_vars["decil_renda"] = pd.qcut(
            df_vars["renda_domiciliar_pc"], 10, labels=False, duplicates="drop"
        ) + 1
    
    dummies_decis = pd.get_dummies(df_vars["decil_renda"], prefix="decil", drop_first=True)
    df_vars = pd.concat([df_vars, dummies_decis], axis=1)
    
    print(f"  Dummies criadas: {list(dummies_decis.columns)[:3]}...")
    
    # SELEÇÃO FINAL
    print("\n[7] Seleção final de variáveis...")
    
    variaveis_explicativas = [
        "idade_25_34", "idade_35_44", "idade_45_54", "idade_65_mais",
        "vive_com_companheiro", "tem_filhos", "trabalha",
        "anos_estudo", "urbano"
    ] + list(dummies_decis.columns)
    
    # LIMPEZA
    faltantes_antes = df_vars[variaveis_explicativas].isna().sum().sum()
    df_vars = df_vars.dropna(subset=variaveis_explicativas)
    faltantes_depois = df_vars[variaveis_explicativas].isna().sum().sum()
    
    print(f"  NaN removidos: {faltantes_antes}")
    print(f"  Observações retidas: {len(df_vars):,}")
    
    X_matriz = df_vars[variaveis_explicativas].astype(float)
    
    assert X_matriz.isna().sum().sum() == 0, "Erro: ainda há NaN em X"
    assert not any(X_matriz.dtypes == "object"), "Erro: X contém colunas object"
    
    print(f"\n✓ Variáveis explicativas construídas!")
    print(f"  Dimensão: {X_matriz.shape}")
    
    return df_vars, X_matriz, variaveis_explicativas


df, X_matriz_sem_constante, lista_vars_explicativas = build_explanatory_variables(df_limpo)


CONSTRUÇÃO DE VARIÁVEIS EXPLICATIVAS

[1] Faixas etárias...
  25-34: 6340; 35-44: 6006
  45-54: 4672; 65+: 3171

[2] Estrutura familiar...
  Vive com companheiro: 13730
  Tem filhos: 11853

[3] Inserção no mercado de trabalho...
  Trabalha (ocupada): 12321.0

[4] Escolaridade...
  Média: 8.55 anos

[5] Localização...
  Urbana: 20126

[6] Decis de renda...
  Dummies criadas: ['decil_2.0', 'decil_3.0', 'decil_4.0']...

[7] Seleção final de variáveis...
  NaN removidos: 14265
  Observações retidas: 10,270

✓ Variáveis explicativas construídas!
  Dimensão: (10270, 18)


## 5. DIAGNÓSTICO PRÉ-MODELO

In [7]:
# Alinhamento final de X e y
indices_comuns = X_matriz_sem_constante.index.intersection(y_forma_pagamento.index)

X_final = X_matriz_sem_constante.loc[indices_comuns].copy()
y_final = y_forma_pagamento.loc[indices_comuns].copy()

print(f"\nAlinhamento de índices:")
print(f"  X: {X_final.shape[0]:,} obs × {X_final.shape[1]} vars")
print(f"  y: {len(y_final):,} obs")
print(f"  ✓ Alinhados: {X_final.shape[0] == len(y_final)}")

# Diagnóstico
print("\n" + "="*70)
print("DIAGNÓSTICO PRÉ-MODELO")
print("="*70)

print(f"\n[✓] Dimensões:")
print(f"  Amostra final: {len(y_final):,} observações")
print(f"  Variáveis: {X_final.shape[1]}")

print(f"\n[✓] Valores faltantes:")
print(f"  NaN em X: {X_final.isna().sum().sum()}")
print(f"  NaN em y: {y_final.isna().sum()}")

print(f"\n[✓] Tipos de dados:")
print(f"  X: {X_final.dtypes.unique()}")
print(f"  y: {y_final.dtype}")

print(f"\n[✓] Distribuição de y:")
for cat, label in enumerate(["SUS", "Plano", "Pagou"]):
    freq = (y_final == cat).sum()
    pct = 100 * freq / len(y_final)
    print(f"  {cat} ({label}): {freq:,} ({pct:.1f}%)")

assert X_final.shape[0] == len(y_final), "Erro: X e y desalinhados"
assert X_final.isna().sum().sum() == 0, "Erro: NaN em X"
assert y_final.isna().sum() == 0, "Erro: NaN em y"
assert y_final.nunique() == 3, "Erro: y deve ter 3 categorias"

print("\n✓ DADOS PRONTOS PARA MODELAGEM!")


Alinhamento de índices:
  X: 10,270 obs × 18 vars
  y: 10,270 obs
  ✓ Alinhados: True

DIAGNÓSTICO PRÉ-MODELO

[✓] Dimensões:
  Amostra final: 10,270 observações
  Variáveis: 18

[✓] Valores faltantes:
  NaN em X: 0
  NaN em y: 0

[✓] Tipos de dados:
  X: [dtype('float64')]
  y: int64

[✓] Distribuição de y:
  0 (SUS): 6,855 (66.7%)
  1 (Plano): 1,879 (18.3%)
  2 (Pagou): 1,536 (15.0%)

✓ DADOS PRONTOS PARA MODELAGEM!


## 6. ESTIMAÇÃO DO LOGIT MULTINOMIAL

**Especificação:**
$$\Pr(Y_i = j | \mathbf{X}_i) = \frac{\exp(\mathbf{X}_i \boldsymbol{\beta}_j)}{\sum_{k=0}^{2} \exp(\mathbf{X}_i \boldsymbol{\beta}_k)}$$

onde:
- $Y_i \in \{0, 1, 2\}$ é a forma de pagamento
- $j \in \{1, 2\}$ são as categorias (Plano, Pagou)
- Categoria 0 (SUS) é referência com $\boldsymbol{\beta}_0 = 0$
- Método: Newton com máximo 100 iterações

In [10]:
print("\n" + "="*70)
print("ESTIMAÇÃO DO MODELO LOGIT MULTINOMIAL (MLE CLÁSSICO)")
print("="*70)

X_modelo = sm.add_constant(X_final, has_constant="add").astype(float)
y_modelo = y_final.copy()

modelo_estimado = sm.MNLogit(
    y_modelo,
    X_modelo
).fit(
    method="newton",
    maxiter=100,
    disp=False
)

print("✓ Estimação concluída")
print(f"✓ Convergência: {modelo_estimado.mle_retvals['converged']}")
print(f"✓ Iterações: {modelo_estimado.mle_retvals['iterations']}")

print("\n" + "="*70)
print("SUMÁRIO")
print("="*70)
print(modelo_estimado.summary())



ESTIMAÇÃO DO MODELO LOGIT MULTINOMIAL (MLE CLÁSSICO)
✓ Estimação concluída
✓ Convergência: True
✓ Iterações: 7

SUMÁRIO
                          MNLogit Regression Results                          
Dep. Variable:      y_forma_pagamento   No. Observations:                10270
Model:                        MNLogit   Df Residuals:                    10232
Method:                           MLE   Df Model:                           36
Date:                Tue, 27 Jan 2026   Pseudo R-squ.:                  0.1274
Time:                        17:14:36   Log-Likelihood:                -7749.4
converged:                       True   LL-Null:                       -8881.1
Covariance Type:            nonrobust   LLR p-value:                     0.000
 y_forma_pagamento=1       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   -5.6214      0.278    -20.200      0.000     